# Scikit-Learn Pipeline Practice

**Topic:** scikit-learn Workflow &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `train_df` (2,500 rows) and `test_df` (900 rows), a subscription-churn table:
  numeric `tenure_months`, `monthly_fee`, `support_calls` (with `NaN`s), and categorical `plan` and `payment_method` (strings).
- `train_df["churned"]` is the 0/1 label, and the test labels are kept apart from `test_df` (the evaluation cell scores your predictions against them).

## Your task

1. Build a `ColumnTransformer`: numeric branch (impute + scale), categorical branch (`OneHotEncoder(handle_unknown="ignore")`).
2. Wrap it in a `Pipeline` with a classifier and cross-validate the whole object.
3. Fit on the full training data and set `predictions` for `test_df` (0/1).

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np
import pandas as pd

def make(n, seed, with_label):
    r = np.random.default_rng(seed)
    tenure = r.gamma(3.0, 9.0, n)
    fee = r.uniform(8, 80, n)
    calls = r.poisson(1.6, n).astype(float)
    plan = r.choice(["basic", "plus", "pro"], n, p=[0.5, 0.3, 0.2])
    pay = r.choice(["card", "bank", "paypal", "crypto"], n)
    logit = (
        -0.15 * tenure + 0.09 * fee + 1.65 * calls
        + np.select([plan == "basic", plan == "plus"], [1.8, 0.3], default=-1.2)
        + (pay == "crypto") * 2.1 - 3.6
    )
    y = (1 / (1 + np.exp(-logit)) > r.uniform(0, 1, n)).astype(int)
    df = pd.DataFrame({
        "tenure_months": tenure.round(1), "monthly_fee": fee.round(2),
        "support_calls": calls, "plan": plan, "payment_method": pay,
    })
    for col in ["tenure_months", "monthly_fee", "support_calls"]:
        df.loc[r.random(n) < 0.07, col] = np.nan
    if with_label:
        df["churned"] = y
    return df, y

train_df, _ = make(2500, 151, True)
test_df, _y_test = make(900, 152, False)
print(train_df.dtypes)
train_df.head()


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.75

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(_y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

One `ColumnTransformer` routes numeric columns through impute+scale and categorical columns through one-hot encoding. Wrapping it in a `Pipeline` with logistic regression makes cross-validation and the final fit leak-proof one-liners.

```python
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ["tenure_months", "monthly_fee", "support_calls"]
categorical = ["plan", "payment_method"]

prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))])

X = train_df.drop(columns=["churned"])
y = train_df["churned"]
print("CV accuracy:", cross_val_score(model, X, y, cv=5).mean().round(3))

model.fit(X, y)
predictions = model.predict(test_df)
```
